## Instalando bibliotecas

In [1]:
!pip install transformers datasets peft accelerate torch

## Importando bibliotecas

In [2]:
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)
import torch

/home/jessica/Área de trabalho/topicosAvancadosEmIAAAvaliacao02/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Preparando Dataset

In [3]:
def convert_to_hf_format(example):
    """Combina instrução e saída em um único texto."""
    return {
        "text": f"Instruction: {example['Instruction']}\nOutput: {example['Output']}"
    }

# Carrega o arquivo JSON Lines
dataset = load_dataset('json', data_files='dataset_sem_curadoria.jsonl')
# Aplica a conversão
dataset = dataset.map(convert_to_hf_format)
# Divide em treino e teste
dataset = dataset["train"].train_test_split(test_size=0.2)
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['Instruction', 'Output', 'text'],
        num_rows: 89
    })
    test: Dataset({
        features: ['Instruction', 'Output', 'text'],
        num_rows: 23
    })
})


## Carregar o Modelos Pré-Treinado e o Tokenizador

In [4]:
model_name = "nicholasKluge/TeenyTinyLlama-460m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForCausalLM.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Modelo carregado: {model_name}")

Loading weights: 100%|██████████| 219/219 [00:04<00:00, 44.02it/s]


Modelo carregado: nicholasKluge/TeenyTinyLlama-460m


## Testando Modelo Crú

In [5]:
def generate_response(model, tokenizer, instruction, input_text=""):
    """Gera uma resposta a partir de uma instrução, usando o modelo fornecido."""
    if input_text:
        prompt = f"Instruction: {instruction}\nInput: {input_text}\nOutput:"
    else:
        prompt = f"Instruction: {instruction}\nOutput:"
    
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=150,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,          # ativa amostragem para usar temperatura
        temperature=0.7
    )
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extrai apenas a parte após "Output:"
    resposta = full_output.split("Output:")[-1].strip()
    return resposta

# Exemplo de instrução (deve existir no dataset.jsonl)
test_instruction = "Qual é a prioridade ao inspecionar o local de trabalho antes de usar a máquina?"

print("=== ANTES DO FINE-TUNING ===")
print(f"Instrução: {test_instruction}")
print(f"Resposta base: {generate_response(base_model, tokenizer, test_instruction)}")

=== ANTES DO FINE-TUNING ===
Instrução: Qual é a prioridade ao inspecionar o local de trabalho antes de usar a máquina?
Resposta base: ----------- a revisão 
 ------- a descrição 
 ------------------------------------------- 
 * O uso da máquina não foi detectado, mas é importante para garantir que os trabalhadores permaneçam em seus locais de trabalho após a inspeção.* 
 * Um funcionário deve ser capaz de acompanhar as tarefas e verificar seu bem-estar enquanto estiver no escritório.**** 
 * É vital para um empregador identificar e corrigir quaisquer problemas com sua máquina. (**) 
 
 A análise de desempenho visa determinar a eficiência, precisão e satisfação dos funcionários com uma empresa ou organização. Ele analisa várias métricas, como produtividade, retenção de funcionários, engajamento do cliente e qualidade geral de suas operações de negócios. 
 A análise de desempenho pode envolver vários aspectos, incluindo: 
>> * Análise quantitativa


## Tokenização do Dataset

In [6]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)
print("Dataset tokenizado:", tokenized_datasets)

Map: 100%|██████████| 23/23 [00:00<00:00, 1822.47 examples/s]

Dataset tokenizado: DatasetDict({
    train: Dataset({
        features: ['Instruction', 'Output', 'text', 'input_ids', 'attention_mask'],
        num_rows: 89
    })
    test: Dataset({
        features: ['Instruction', 'Output', 'text', 'input_ids', 'attention_mask'],
        num_rows: 23
    })
})


## Preparar o Modelo para LoRA

In [7]:
# A partir daqui, usaremos uma cópia do modelo base para o fine-tuning
model = base_model  # poderia também ser uma nova instância
model = prepare_model_for_kbit_training(model)

## Verificando Camadas Target

In [8]:
import torch.nn as nn

for name, module in base_model.named_modules():
    if isinstance(module, nn.Linear):
        print(name)

model.layers.0.self_attn.q_proj
model.layers.0.self_attn.k_proj
model.layers.0.self_attn.v_proj
model.layers.0.self_attn.o_proj
model.layers.0.mlp.gate_proj
model.layers.0.mlp.up_proj
model.layers.0.mlp.down_proj
model.layers.1.self_attn.q_proj
model.layers.1.self_attn.k_proj
model.layers.1.self_attn.v_proj
model.layers.1.self_attn.o_proj
model.layers.1.mlp.gate_proj
model.layers.1.mlp.up_proj
model.layers.1.mlp.down_proj
model.layers.2.self_attn.q_proj
model.layers.2.self_attn.k_proj
model.layers.2.self_attn.v_proj
model.layers.2.self_attn.o_proj
model.layers.2.mlp.gate_proj
model.layers.2.mlp.up_proj
model.layers.2.mlp.down_proj
model.layers.3.self_attn.q_proj
model.layers.3.self_attn.k_proj
model.layers.3.self_attn.v_proj
model.layers.3.self_attn.o_proj
model.layers.3.mlp.gate_proj
model.layers.3.mlp.up_proj
model.layers.3.mlp.down_proj
model.layers.4.self_attn.q_proj
model.layers.4.self_attn.k_proj
model.layers.4.self_attn.v_proj
model.layers.4.self_attn.o_proj
model.layers.4.mlp.g

## Configurar e Injetar LoRA

In [9]:
lora_config = LoraConfig(
    r=16,                    # rank da decomposição
    lora_alpha=32,           # escala alpha
    target_modules=["q_proj", "k_proj"],  # camadas alvo , "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    inference_mode=False     # False = modo treinamento
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,572,864 || all params: 469,812,224 || trainable%: 0.3348


## Data Collator (Modelagem Causal)

In [10]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

## Argumentos de Treinamento (Preparando Hiperparâmetros)

In [14]:
training_args = TrainingArguments(
    output_dir="./modelos_treinados",          # diretório de saída
    eval_strategy="steps",
    eval_steps=100,
    learning_rate=1e-3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=10,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    load_best_model_at_end=True,
    report_to="none",               # desabilita logging para wandb/mlflow
)

## Iniciar Treino

In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
)

## Treinar Modelo

In [16]:
trainer.train()

Step,Training Loss,Validation Loss
100,2.293225,2.645319
200,1.636017,2.888636
230,1.523324,2.950526


/home/jessica/Área de trabalho/topicosAvancadosEmIAAAvaliacao02/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/home/jessica/Área de trabalho/topicosAvancadosEmIAAAvaliacao02/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=230, training_loss=2.1986738163491952, metrics={'train_runtime': 6881.1623, 'train_samples_per_second': 0.129, 'train_steps_per_second': 0.033, 'total_flos': 298728467988480.0, 'train_loss': 2.1986738163491952, 'epoch': 10.0})

## Salvar o Modelo Ajustado e o Tokenizador

In [18]:
model.save_pretrained("lora_lhama_causal_finetuned_model")
tokenizer.save_pretrained("lhama_tokenizer")

('lhama_tokenizer/tokenizer_config.json', 'lhama_tokenizer/tokenizer.json')

## Teste pós fine-tuning

In [19]:
# Carrega o modelo fine-tunado (apenas os adaptadores LoRA)
finetuned_model = AutoModelForCausalLM.from_pretrained("lora_lhama_causal_finetuned_model")
finetuned_tokenizer = AutoTokenizer.from_pretrained("lhama_tokenizer")

# Garante que o token de padding esteja definido
if finetuned_tokenizer.pad_token is None:
    finetuned_tokenizer.pad_token = finetuned_tokenizer.eos_token

Loading weights: 100%|██████████| 96/96 [00:00<00:00, 5767.51it/s]


In [ ]:
print("=== DEPOIS DO FINE-TUNING ===")
print(f"Instrução: {test_instruction}")
print(f"Resposta ajustada: {generate_response(finetuned_model, finetuned_tokenizer, test_instruction)}")

=== DEPOIS DO FINE-TUNING ===
Instrução: Qual o modelo da maquina Genie?
Resposta ajustada: Instruction: Qual o modelo da maquina Genie. 
> Commandos: Instrução: Quais os modelos de máquina disponíveis para o usuário? Instruções: Qual a instrução específica para o usuário? Contexto:
a presente pesquisa teve como objetivo desenvolver um material de apoio educacional à formação inicial e continuada de professores do ensino fundamental ii na modalidade de ensino a distância por meio do suporte online  sped. foi realizado um levantamento bibliográfico sobre a temática abordado com base em bibliografia clássica e popular e a realização de entrevistas em profundidade onde foram abordados temas como: metodologia, metodologia; uso dos recursos didáticos: concepção, construção e distribuição de conteúdos; práticas pedagógicas desenvolvidas no sited; aprendizagem colaborativa e desenvolvimento
